# Mission 1 학습 노트북 — 신고자 성별 분류 (팀 eigensinn)

**최종 모델**: 신고자 음성의 52차원 음향 특징 + WavLM x-vector(512차원, 20초 창 최대 4개 평균) = 564차원을
`StandardScaler → LogisticRegression`으로 분류합니다. 최종 체크포인트는 **Training 29,200통화 전체**로 학습합니다.

### 규정 준수
- 학습 데이터는 **Training 폴더만** 사용합니다. Validation 데이터는 이 노트북의 어떤 학습 단계에도 들어가지 않습니다.
- 라벨 JSON은 `startAt`, `endAt`, `Speaker`(입력)와 `gender`(정답)만 읽습니다.
- `microsoft/wavlm-base-plus-sv`는 공개 사전학습 모델이며 **고정 revision**을 그대로 사용합니다(미세조정 없음).

### 이 노트북이 하는 일 / 하지 않는 일
| 단계 | 내용 |
|---|---|
| 1 | 특징 추출 실행 기록(별도 스크립트, Colab T4) 확인 |
| 2 | 저장된 Training 특징 캐시로 564차원 입력 구성, 정답 라벨 읽기 |
| 3 | 80:20 분할(seed 42)로 학습해 holdout 정확도 재현 (선택·튜닝 없음) |
| 4 | **최종 모델: 전체 29,200통화로 재학습**, 체크포인트 저장 |
| 5 | 제출 체크포인트(`m1_model.pt`)와 제출 코드(`mission1_model.py`)로 일부 Training 통화의 특징을 다시 뽑아 캐시와 일치하는지 확인 |

**투명성 메모**: 29,200통화의 WavLM 특징 추출은 GPU로 약 5시간이 걸려 이 노트북이 아니라 별도 스크립트
(`train/extract_features_colab.py`, Colab T4)로 실행했고, 결과를 2,000통화 단위 `.npz` 캐시로 저장했습니다.
이 노트북은 그 캐시를 읽습니다. 제출 가중치 `m1_model.pt`(헤드 + WavLM)는 `train/build_model_pt.py`가 이 노트북의 학습 결과와 WavLM 스냅샷을 한 파일로 묶은 것입니다. 5단계에서 같은 추출 코드를 일부 통화에 다시 돌려 캐시와 같은 값이 나오는 것을 보입니다.

In [1]:
# 설정과 라이브러리. 경로는 환경변수로 바꿀 수 있다 (기본값은 팀 PC의 위치).
import glob, hashlib, json, os, sys, time, unicodedata
from pathlib import Path

import joblib
import numpy as np
import sklearn
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

HERE = Path.cwd()                     # 이 노트북이 있는 submission/mission1
sys.path.insert(0, str(HERE))
import mission1_model as m            # 제출 코드와 같은 특징 추출 함수

REPO = HERE.parents[1]
RUN_DIR = Path(os.environ.get("M1_RUN_DIR", "G:/내 드라이브/dcc_runs/lab/M1-XVEC-002"))
TRAIN_LABELS = Path(os.environ.get("M1_TRAIN_LABELS", REPO / "data/raw_merged/Training/2.라벨링데이터/TL_서울_구급"))
TRAIN_AUDIO = Path(os.environ.get("M1_TRAIN_AUDIO", REPO / "data/raw_merged/Training/1.원천데이터/TS_서울_구급"))
OUT_DIR = HERE / "nb_outputs"
OUT_DIR.mkdir(exist_ok=True)
SEED, N_CHECK = 42, 50
print("sklearn", sklearn.__version__, "| numpy", np.__version__)

sklearn 1.8.0 | numpy 2.4.4


## 1. 특징 추출 실행 기록 (별도 스크립트, Colab T4)

In [2]:
# 특징 추출 실행의 설정과 내부 holdout 결과를 그대로 출력한다.
print(json.dumps(json.loads((RUN_DIR / "config.json").read_text()), indent=1, ensure_ascii=False))
heads = json.loads((RUN_DIR / "metrics.json").read_text())["metrics"]["heads"]
for name, h in heads.items():
    print(f"{name:26s} holdout acc {h['accuracy']:.4f}  recall F {h['recall_F']:.4f}  M {h['recall_M']:.4f}  (n={h['n_samples']})")
print()
print("\n".join((RUN_DIR / "run.log").read_text(errors="ignore").splitlines()[-6:]))

{
 "exp_id": "M1-XVEC-002",
 "model": "microsoft/wavlm-base-plus-sv",
 "revision": "1bfd64eca136543feb28c5ffaf05381c6af33121",
 "seed": 42,
 "chunk_size": 2000,
 "expected_calls": 29200,
 "sample_rate": 16000,
 "max_seconds": 80,
 "window_seconds": 20,
 "max_windows": 4,
 "tail_seconds": 3,
 "min_seconds": 0.5,
 "holdout_ratio": 0.2,
 "schema": 2
}
a_acoustic_52d             holdout acc 0.9502  recall F 0.9582  M 0.9411  (n=5840)
b_first_combined_564d      holdout acc 0.9738  recall F 0.9775  M 0.9696  (n=5840)
c_mean_combined_564d       holdout acc 0.9772  recall F 0.9804  M 0.9736  (n=5840)



Loading weights: 100%|██████████| 266/266 [00:00<00:00, 752.97it/s]
/usr/local/lib/python3.13/dist-packages/torch/nn/functional.py:6409: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(
Completed feature chunk 26000:28000
Completed feature chunk 28000:29200


## 2. 입력 구성: Training 특징 캐시 + 정답 라벨
행 순서는 특징 추출 때와 같게 라벨 JSON 파일명(NFC·소문자 정규화)을 정렬한 순서입니다.
`gender`(F/M)만 정답으로 읽습니다.

In [3]:
# 정답 라벨 읽기 (Training 라벨 JSON의 gender)
def sort_key(path):
    return unicodedata.normalize("NFC", path.stem).casefold()

json_files = sorted(TRAIN_LABELS.rglob("*.json"), key=sort_key)
genders = [json.loads(p.read_text(encoding="utf-8-sig"))["gender"] for p in json_files]
y = np.array([g == "M" for g in genders]).astype(int)      # F=0, M=1
print("통화 수", len(y), "| 여(F)", int((y == 0).sum()), "| 남(M)", int((y == 1).sum()))
assert len(y) == 29200 and set(genders) == {"F", "M"}

통화 수 29200 | 여(F) 15542 | 남(M) 13658


In [4]:
# 특징 캐시 읽기 → 564차원 입력 만들기
chunks = [np.load(f) for f in sorted(glob.glob(str(RUN_DIR / "features" / "chunk_*.npz")))]
acoustic = np.concatenate([c["acoustic"] for c in chunks])                 # (N, 52)
window_x = np.concatenate([c["window_xvectors"] for c in chunks])          # (N, 4, 512) 창별 x-vector
window_len = np.concatenate([c["window_lengths"] for c in chunks])         # (N, 4) 창 길이(초)
window_cnt = np.concatenate([c["window_counts"] for c in chunks])          # (N,) 사용한 창 수
def mean_xvector(vectors, lengths):
    # 창별 x-vector를 정규화 → 창 길이 가중 평균 → 다시 정규화 (float64 정밀도로 계산)
    vectors = vectors / np.linalg.norm(vectors, axis=1, keepdims=True)
    mean = (vectors * lengths[:, None]).sum(0) / lengths.sum()
    return mean / np.linalg.norm(mean)

mean_x = np.stack([mean_xvector(v[:n], l[:n]) for v, l, n in zip(window_x, window_len, window_cnt)])
X = np.hstack([acoustic, mean_x])          # float64 (추론 코드도 float64 특징을 만든다)
print("X", X.shape, "| 통화당 창 수 분포", dict(zip(*np.unique(window_cnt, return_counts=True))))
assert X.shape == (len(y), m.N_FEATURES) and np.isfinite(X).all()

X (29200, 564) | 통화당 창 수 분포 {np.int64(1): np.int64(7480), np.int64(2): np.int64(15285), np.int64(3): np.int64(4981), np.int64(4): np.int64(1454)}


## 3. 80:20 분할 재현 (선택·튜닝 없음)
특징 추출 실행이 사용한 분할(통화 단위, 성별 층화, seed 42)을 똑같이 만들어 holdout 정확도를 다시 계산합니다.
라벨 순서가 맞는지 확인하는 점검이기도 합니다.

In [5]:
def make_pipeline():
    return Pipeline([("scaler", StandardScaler()),
                     ("clf", LogisticRegression(max_iter=1000, random_state=SEED))])

train_idx, holdout_idx = train_test_split(np.arange(len(y)), test_size=0.2, random_state=SEED, stratify=y)
saved = json.loads((RUN_DIR / "split.json").read_text())
print("분할이 특징 추출 실행과 같은가:", train_idx.tolist() == saved["train"] and holdout_idx.tolist() == saved["holdout"])

t0 = time.time()
pipe80 = make_pipeline().fit(X[train_idx], y[train_idx])
acc = (pipe80.predict(X[holdout_idx]) == y[holdout_idx]).mean()
print(f"학습 {len(train_idx)}통화 / holdout {len(holdout_idx)}통화 → holdout 정확도 {acc:.4f}  (기록: 0.9772)  [{time.time()-t0:.1f}초]")

분할이 특징 추출 실행과 같은가: True


학습 23360통화 / holdout 5840통화 → holdout 정확도 0.9774  (기록: 0.9772)  [3.2초]


## 4. 최종 모델: Training 29,200통화 전체로 재학습
같은 구성(`StandardScaler` + `LogisticRegression`, 하이퍼파라미터 탐색 없음)으로 holdout을 포함한 Training 전체를 학습합니다.
Validation은 사용하지 않습니다.

In [6]:
t0 = time.time()
final = make_pipeline().fit(X, y)
print(f"전체 {len(y)}통화 학습 완료 [{time.time()-t0:.1f}초] | 학습 데이터 정확도 {(final.predict(X) == y).mean():.4f}")

ckpt = OUT_DIR / "m1_xvec002_refit_full.ckpt"
joblib.dump(final, ckpt)
digest = hashlib.sha256(ckpt.read_bytes()).hexdigest()
print("저장:", ckpt.name, "| SHA256", digest)

# 제출 체크포인트(m1_model.pt 안의 헤드)와 같은 모델인지 비교: 학습된 파라미터를 하나씩 대조한다.
# (파일 바이트는 저장 방식에 따라 달라질 수 있어 해시 대신 파라미터를 비교한다.)
import torch
submitted = HERE / "m1_model.pt"
if submitted.exists():
    head = {k: v.numpy() for k, v in torch.load(submitted, map_location="cpu", weights_only=True)["head"].items()}
    pairs = {"표준화 평균": (head["scaler_mean"], final[0].mean_), "표준화 표준편차": (head["scaler_scale"], final[0].scale_),
             "로지스틱 계수": (head["coef"], final[-1].coef_), "절편": (head["intercept"], final[-1].intercept_)}
    for name, (a, b) in pairs.items():
        print(f"{name}: 제출 체크포인트와 완전 일치 = {np.array_equal(a, b)}")
    submitted_pred = np.array([m.classify(head, x) for x in X])      # 제출 코드가 쓰는 numpy 헤드
    print("Training 예측 일치율(제출 헤드 vs 방금 학습한 모델):", (submitted_pred == final.predict(X)).mean())

전체 29200통화 학습 완료 [5.7초] | 학습 데이터 정확도 0.9850
저장: m1_xvec002_refit_full.ckpt | SHA256 00845d2396cb09e0b532890fbb4839b94d32e49fba350d93c0c85e59f1fe741a


표준화 평균: 제출 체크포인트와 완전 일치 = True
표준화 표준편차: 제출 체크포인트와 완전 일치 = True
로지스틱 계수: 제출 체크포인트와 완전 일치 = True
절편: 제출 체크포인트와 완전 일치 = True
Training 예측 일치율(제출 헤드 vs 방금 학습한 모델): 1.0


## 5. 제출 추론 코드와 캐시 비교
`mission1_model.extract_features`(제출 코드)로 Training 통화 일부(50통화)의 564차원 특징을 처음부터 다시 만들어,
학습에 쓴 캐시 값과 같은지 확인합니다. 추론 때와 학습 때 입력이 같다는 점을 보이는 점검입니다.

In [7]:
head_arrays, processor, wavlm, torch = m.load_model(HERE / "m1_model.pt")   # 제출 체크포인트(.pt 한 파일)
picks = np.linspace(0, len(json_files) - 1, N_CHECK).astype(int)
diff_ac, diff_xv, agree, t0 = [], [], [], time.time()
for i in picks:
    stem = json_files[i].stem
    row = m.extract_features(TRAIN_AUDIO / f"{stem}.wav", json_files[i], processor, wavlm, torch)
    diff_ac.append(np.abs(row[:52] - X[i, :52]).max())
    diff_xv.append(np.abs(row[52:] - X[i, 52:]).max())
    agree.append(m.classify(head_arrays, row) == int(final.predict(X[i:i + 1])[0]))
print(f"{N_CHECK}통화 재추출 [{time.time()-t0:.0f}초, 통화당 {(time.time()-t0)/N_CHECK:.1f}초]")
print(f"52D 음향 특징 최대 오차 {max(diff_ac):.2e} | x-vector 512D 최대 오차 {max(diff_xv):.2e}")
print(f"재추출 입력과 캐시 입력의 예측 일치 {sum(agree)}/{N_CHECK}")

C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


...

C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


C:\Users\sooyah\AppData\Local\Programs\Python\Python311\Lib\site-packages\torch\nn\functional.py:6441: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


50통화 재추출 [188초, 통화당 3.8초]
52D 음향 특징 최대 오차 6.10e-05 | x-vector 512D 최대 오차 9.47e-07
재추출 입력과 캐시 입력의 예측 일치 50/50


## 요약
- 최종 모델 = 52D 음향 + 다중 창 평균 x-vector(564D) → StandardScaler + LogisticRegression, **Training 29,200통화 전체**로 학습.
- 내부 holdout(Training 20%) 정확도는 위 3단계에서 재현했고(약 0.977), 최종 모델은 holdout을 포함해 다시 학습했으므로 이 값으로 평가하지 않습니다.
- 공식 Validation 평가 결과는 `README.md`에 있습니다(학습에 사용하지 않고 평가에만 사용).